# Normalise entity names

Many subjects and objects in `ALL_relations.json` carry punctuation left over from the source sentence (for example `dielectric barrier.` or `quartz lens,`). Spaces and hyphens inside names also vary (`self-pulsing operation`, `fluid model`). This notebook cleans the `subject` and `object` fields of every relation:

1. Strips punctuation from the start and end of each name: `.` `,` `;` `:` `!` `?` quotes, dashes and stray brackets. A bracket is only removed when it has no partner, so `μs).` becomes `μs` but `dielectric barrier(s)` keeps its brackets.
2. Replaces each run of spaces and hyphens inside the name with one underscore.

`+`, `*` and `%` at the ends are kept by default because they carry meaning here (`Ar +` is an ion, `Ar *` an excited state, `vol%` a unit). Clear `KEEP_AT_ENDS` to strip them too.

Run the cells in order from the `kg_viewer2` folder. Running it again is safe: once every name is clean, nothing changes.

In [1]:
import json
import re
import shutil
import string
import subprocess
import sys
from collections import Counter
from pathlib import Path

DATA_FILE = Path('ALL_relations.json')
BACKUP_FILE = Path('ALL_relations.entities.backup.json')
ENTITY_FIELDS = ('subject', 'object')

with DATA_FILE.open(encoding='utf-8-sig') as stream:
    relations = json.load(stream)

print(f'Loaded {len(relations)} relations')
print(f'Unique entities before: {len({r[f] for r in relations for f in ENTITY_FIELDS})}')

Loaded 2077 relations
Unique entities before: 2034


In [2]:
# Characters that can be stripped from either end of a name.
EDGE_PUNCTUATION = set(string.punctuation) | set('®©™—–…“”‘’«»·')
# Edge characters that carry meaning and are kept.
KEEP_AT_ENDS = set('+*%')
OPENING, CLOSING = '([{', ')]}'


def is_unmatched(text, char):
    # A bracket is stray when the name has more of it than of its partner.
    if char in OPENING:
        partner = CLOSING[OPENING.index(char)]
        return text.count(char) > text.count(partner)
    partner = OPENING[CLOSING.index(char)]
    return text.count(char) > text.count(partner)


def strippable(text, char):
    if char.isspace():
        return True
    if char in OPENING or char in CLOSING:
        return is_unmatched(text, char)
    return char in EDGE_PUNCTUATION and char not in KEEP_AT_ENDS


def normalize(entity):
    text = entity.strip()
    while text and strippable(text, text[-1]):
        text = text[:-1]
    while text and strippable(text, text[0]):
        text = text[1:]
    # Each run of spaces and hyphens becomes one underscore.
    return re.sub(r'[\s\-]+', '_', text)


changes = Counter()
for relation in relations:
    for field in ENTITY_FIELDS:
        old = relation[field]
        new = normalize(old)
        if new != old:
            changes[(old, new)] += 1
            relation[field] = new

print(f'Fields changed: {sum(changes.values())}')
print(f'Distinct entities rewritten: {len(changes)}')
for (old, new), count in sorted(changes.items()):
    print(f'  {old!r} -> {new!r}  ({count} fields)')

Fields changed: 3342
Distinct entities rewritten: 1720
  '+ O 2 ions' -> '+_O_2_ions'  (2 fields)
  '0-0 transitions' -> '0_0_transitions'  (1 fields)
  '0.1 vol% O 2 in N 2' -> '0.1_vol%_O_2_in_N_2'  (10 fields)
  '0D model' -> '0D_model'  (1 fields)
  '0D modelling of reaction kinetics' -> '0D_modelling_of_reaction_kinetics'  (1 fields)
  '0D modelling of the reaction kinetics' -> '0D_modelling_of_the_reaction_kinetics'  (1 fields)
  '1 vol % O 2 in N 2' -> '1_vol_%_O_2_in_N_2'  (1 fields)
  '16 memory segments' -> '16_memory_segments'  (1 fields)
  '1D fluid simulation' -> '1D_fluid_simulation'  (3 fields)
  '1D fluid simulation.' -> '1D_fluid_simulation'  (2 fields)
  '1D simulation.' -> '1D_simulation'  (1 fields)
  '2D model' -> '2D_model'  (1 fields)
  '2D models' -> '2D_models'  (2 fields)
  '2D simulations' -> '2D_simulations'  (1 fields)
  '5 vol% N2O in N2' -> '5_vol%_N2O_in_N2'  (1 fields)
  '520 reactions' -> '520_reactions'  (1 fields)
  'AC high voltage' -> 'AC_high_volt

In [3]:
unique_after = sorted({r[f] for r in relations for f in ENTITY_FIELDS})
assert all(unique_after), 'An entity became empty after cleaning'
assert not any(re.search(r'[\s\-]', e) for e in unique_after), 'An entity still contains a space or hyphen'
print(f'Unique entities after: {len(unique_after)}')

Unique entities after: 1913


In [7]:
# Keep a copy of the file as it was before this notebook first ran.
if not BACKUP_FILE.exists():
    shutil.copyfile(DATA_FILE, BACKUP_FILE)
    print(f'Backup written to {BACKUP_FILE}')

# Same formatting as the original file, so only the entity lines change.
with DATA_FILE.open('w', encoding='utf-8', newline='') as stream:
    json.dump(relations, stream, ensure_ascii=False, indent=2)
    stream.write('')
print(f'Saved {DATA_FILE}')

Backup written to ALL_relations.entities.backup.json
Saved ALL_relations.json


In [9]:
# Rebuild data.js so the viewer picks up the cleaned entity names.
result = subprocess.run([sys.executable, 'build_data.py'], capture_output=True, text=True)
print(result.stdout or result.stderr)
result.check_returncode()

Bundled 2077 relations from C:\Users\jaina\OneDrive\Desktop\kg_viewer2\ALL_relations.json into C:\Users\jaina\OneDrive\Desktop\kg_viewer2\data.js

